# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

This optional supporting notebook builds the same public-safe starter feature vector used by the core assignments. The target remains a same-window proxy, so its source fields are kept completely outside the feature matrix.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

REPO_URL="https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR="flyrank-ml-internship"

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"data/raw/content_refresh_anonymized.csv").exists():
            return p
    return None

root=find_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root=Path(REPO_DIR).resolve()
os.chdir(root)

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_proxy"]=df["trend_direction"].str.lower().eq("down").astype(int)

features=[
    "impressions_90d","clicks_90d","sessions_90d","avg_position","ctr",
    "content_age_days","days_since_last_update","word_count",
    "engagement_rate","scroll_rate","days_with_impressions","days_with_sessions"
]
X=df[features].copy()
for col in features:
    X[col]=pd.to_numeric(X[col],errors="coerce")
    X[f"{col}__missing"]=X[col].isna().astype(int)
    X[col]=X[col].fillna(X[col].median())

print("Rows:",len(X))
print("Base features:",len(features))
print("Engineered matrix columns:",X.shape[1])
print("Proxy positives:",int(df["is_declining_proxy"].sum()))


## 2. Feature notes (meaning, missing, categorical, available-when?)

Every model input is an observable numeric signal available in the current starter snapshot. Missing numeric values use median fills plus explicit missingness flags in this audit notebook. IDs are context only. In the future warehouse version, every feature must also be timestamped before the decision point.

In [ ]:
notes=pd.DataFrame({
    "feature":features,
    "missing_rate":[float(df[c].isna().mean()) for c in features],
    "available_before_current_decision":[True]*len(features)
}).sort_values("missing_rate",ascending=False)
print(notes.round(3).to_string(index=False))


## 3. The leakage hunt

The proxy is derived from `trend_direction`; therefore both `trend_direction` and `trend_pct` are direct target siblings and must never enter the feature vector. IDs, product decisions, and future-window fields are also forbidden.

In [ ]:
forbidden={
    "trend_direction","trend_pct","is_declining_label","is_declining_proxy",
    "content_id","client_id","health_score","priority_score","action_type"
}
present=sorted(forbidden.intersection(features))
print("Forbidden fields present in feature list:",present)
print("Target siblings excluded:", {"trend_direction","trend_pct"}.isdisjoint(features))
assert present==[]


## 4. What I excluded and why

- `trend_direction`, `trend_pct`: define the starter proxy, so using them would leak the answer.
- `content_id`, `client_id`: pseudonymous context/grouping keys, not predictive signals.
- product health/action/priority fields: would create circular decision learning.
- any future-window measurements: not knowable at prediction time.
- raw names, URLs, queries, keywords, titles: private and not needed for the ranking task.

In [ ]:
excluded={
    "label_derived":["trend_direction","trend_pct"],
    "context_only":["content_id","client_id"],
    "decision_derived":["health_score","priority_score","action_type"],
    "future_fields":"none allowed",
    "private_raw_fields":"none published"
}
print(excluded)


## Self-check

- [x] Feature vector is built from public-safe observable signals
- [x] Missingness handling is explicit
- [x] Direct label siblings, IDs, decision outputs, and future data are excluded
- [x] No private client names, URLs, or queries are published
- [ ] Notebook execution outputs verified